# Chapter 4: The family of tests

In [1]:
import numpy as np
from expkit.inference.binomial import binom_test_exact
from expkit.inference.normal import one_sample_z, one_sample_t
from expkit.inference.chi2 import goodness_of_fit
from expkit.inference.fisher import fisher_exact_2x2
from expkit.inference.bayes import coin_posterior_conjugate
from expkit.plot.style import apply_style
apply_style()

## Loop A: five tests at four sample sizes

In [2]:
for k, n in [(6, 10), (60, 100), (600, 1000), (6000, 10000)]:
    seq = np.concatenate([np.ones(k), np.zeros(n - k)])
    p_exact = binom_test_exact(k, n).p_value
    p_z = one_sample_z(k, n).p_value
    p_chi = goodness_of_fit(np.array([k, n - k]), expected_p=np.array([0.5, 0.5])).p_value
    table = np.array([[k, n - k], [n // 2, n - n // 2]])
    p_fisher = fisher_exact_2x2(table).p_value
    p_t = one_sample_t(seq, mu_null=0.5).p_value
    post = coin_posterior_conjugate(seq.astype(int))
    print(f'{k:>5}/{n:<6}  exact={p_exact:.4g}  z={p_z:.4g}  chi2={p_chi:.4g}  fisher={p_fisher:.4g}  t={p_t:.4g}  Bayes P(p>0.5)={post.prob_greater_than(0.5):.3f}')

    6/10      exact=0.7539  z=0.5271  chi2=0.5271  fisher=1  t=0.5554  Bayes P(p>0.5)=0.726
   60/100     exact=0.05689  z=0.0455  chi2=0.0455  fisher=0.2007  t=0.04493  Bayes P(p>0.5)=0.977
  600/1000    exact=2.728e-10  z=2.54e-10  chi2=2.54e-10  fisher=8.454e-06  t=1.72e-10  Bayes P(p>0.5)=1.000
 6000/10000   exact=1.74e-89  z=5.507e-89  chi2=5.507e-89  fisher=7.767e-46  t=9.224e-91  Bayes P(p>0.5)=1.000


## Loop B: where normal-approx lies

In [3]:
for n in [10, 50, 100, 1000]:
    diffs = []
    for k in range(0, n + 1):
        diffs.append(abs(one_sample_z(k, n).p_value - binom_test_exact(k, n).p_value))
    print(f'N={n:>5}  max|p_z - p_exact| = {max(diffs):.4f}')

N=   10  max|p_z - p_exact| = 0.2268
N=   50  max|p_z - p_exact| = 0.1104
N=  100  max|p_z - p_exact| = 0.0789


N= 1000  max|p_z - p_exact| = 0.0252


## Loop C: edge cases

In [4]:
for k, n in [(0, 10), (10, 10), (1, 1), (0, 100), (100, 100)]:
    print(f'{k}/{n}: exact={binom_test_exact(k, n).p_value:.4g}  z={one_sample_z(k, n).p_value:.4g}  chi2={goodness_of_fit(np.array([k, n - k]), np.array([0.5, 0.5])).p_value:.4g}')

0/10: exact=0.001953  z=0.001565  chi2=0.001565
10/10: exact=0.001953  z=0.001565  chi2=0.001565
1/1: exact=1  z=0.3173  chi2=0.3173
0/100: exact=1.578e-30  z=1.524e-23  chi2=1.524e-23
100/100: exact=1.578e-30  z=1.524e-23  chi2=1.524e-23


## Loop D: Bayesian credible intervals via the library helper

In [ ]:
for k, n in [(6, 10), (60, 100), (600, 1000), (6000, 10000)]:
    seq = np.concatenate([np.ones(k, dtype=int), np.zeros(n - k, dtype=int)])
    post = coin_posterior_conjugate(seq)
    lo, hi = post.credible_interval(0.95)
    print(f'{k:>5}/{n:<6}  Beta({int(post.alpha)},{int(post.beta)})  mean={post.mean:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]  P(p>0.5)={post.prob_greater_than(0.5):.3f}')